# Reference-qubit purification: benchmark against Gullans & Huse 2020, Fig. 1

Experiment `experiments/purification/purification.py` (issue #185). It reproduces the probe of the measurement-induced transition of M. J. Gullans and D. A. Huse, “Scalable probes of measurement-induced criticality”, PRL **125**, 070606 (2020), [arXiv:1910.00020](https://arxiv.org/abs/1910.00020). The paper's chain length `L` is our `N`, the number of system qubits (issue #235); `L` here is the number of trajectories per run.

The protocol, configured by `conf/purification_gullans_huse.yaml`:

1. the reference qubit `R` starts in a Bell pair with the middle system site, the other system qubits in `|0>`;
2. `t0 = 2N` steps of random two-qubit Clifford gates on the periodic brickwork, without measurement;
3. `2N` more steps, each followed by a Z measurement of every system qubit with probability `p`;
4. the order parameter is `S_Q`, the entropy of `R` at `t = 4N`, averaged over circuits (new gates for every trajectory) and trajectories.

A step is one half-layer of the brickwork: every site gets one gate per step, the paper's time unit. The state is a stabilizer state, simulated with the tableaux of `stim`, which reaches the paper's sizes; `tests/test_stabilizer_probe.py` checks it against the state vector of the library at small `N`. In each trajectory `S_Q` is 0 or 1 bit, so `<S_Q>` is the probability that `R` is still mixed.

The paper finds that the curves of different `N` cross at `p_c = 0.1598(5)`, with `nu = 1.30(5)` from a data collapse (sizes 64 to 256), and `p_c = 0.16(1)` already at small sizes.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what `tests/test_purification_notebook.py` does for a small sweep). Each sweep goes to this notebook's own folder, `outputs/purification_gullans_huse/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

With the default values (`N = 8` to `64`, 11 values of `p`, 2000 circuits per point, 4 runs in parallel) the sweep takes about 15 minutes; `N = 128` adds about half an hour.

In [ ]:
L = os.environ.get("L", "2000")  # trajectories (circuits) per run
N_VALUES = os.environ.get("N_VALUES", "8 16 32 64")  # the paper's L, even
P_VALUES = os.environ.get(
    "P_VALUES", "0.10 0.12 0.14 0.15 0.155 0.16 0.165 0.17 0.18 0.20 0.22"
)
SEEDS = os.environ.get("SEEDS", "0")  # several seeds split the trajectories
JOBS = os.environ.get("JOBS", "4")  # runs in parallel
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "purification_gullans_huse"

if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "P_VALUES": P_VALUES,
        "SEEDS": SEEDS,
        "JOBS": JOBS,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = len(N_VALUES.split()) * len(P_VALUES.split()) * len(SEEDS.split())
    # One bar for the whole sweep; raises CalledProcessError, with the
    # runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/purification/purification.sh"],
        total=n_runs,
        env=env,
        show=("N", "p", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back `S_Q`

One row per `(N, p)`, the seeds pooled: `<S_Q>` at `t = 4N` and its standard error over the trajectories.

In [ ]:
import importlib.util

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_purification", Path("experiments/purification/plot_purification.py")
)
plot_pur = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_pur)

curves, traces = plot_pur.pooled_curves(sweep_dir)
print(f"{'N':>4} {'p':>6} {'<S_Q>':>16}")
for N, (p, y, e) in sorted(curves.items()):
    for q, m, s in zip(p, y, e, strict=True):
        print(f"{N:>4} {q:>6.3f} {m:>7.4f} ± {s:.4f}")

## Fig. 1: crossing and collapse

- (a) `<S_Q>` at `t = 4N` against `p`: the curves of different `N` cross near `p_c` (dotted line, the paper's value).
- (b) The data collapse against `(p - p_c) N^(1/nu)`, with `(p_c, nu)` fitted on `N >= 16` (quality function of Houdayer & Hartmann, errors from a parametric bootstrap). The sizes left out of the fit are pale.
- (c) The crossing of each pair of consecutive sizes against `1 / N`: it drifts towards `p_c` as `N` grows.
- (d) `<S_Q(t)>` at the `p` closest to `p_c`: 1 during the unitary scrambling, then the decay under measurements from `t = 2N`.

The figure, its data and the fit (`fss.json`) are saved in the sweep folder.

In [ ]:
import json

import matplotlib.pyplot as plt

from QELM.utils.plots import save_figure

fig, fss = plot_pur.draw_sweep(sweep_dir)
figure_path = save_figure(fig, sweep_dir, plot_pur.FIGURE_NAME, data={"fss": fss})
(sweep_dir / "fss.json").write_text(json.dumps(fss, indent=2) + "\n")
print(json.dumps({k: v for k, v in fss.items() if k != "paper"}, indent=2))
print(f"figure: {figure_path}")
plt.show()